# HIPAA Privacy Rule-based De-identification on DICOM Dataset

HIPAA provides two methods for de-identification: the "Safe Harbor" method and the "Expert Determination" method. The Safe Harbor method is more straightforward and involves anonymizing/redacting 18 specific types of identifiers from the data.

Here, we will focus on the Safe Harbor method, which includes removing or redacting identifiers such as names, geographic subdivisions smaller than a state, dates directly related to an individual, phone numbers, email addresses, and more.

After de-ID,  the DICOM file will be updated and uploaded to destiny storage and evaluated by AWS services, Rekongnition, Comprehend and Comprehend Medical.

## Setup De-identification Environment

Let's start by setting environment variables for de identification of DICOM file:
1) set local path of DICOM img folder.
2) set source and destiny s3 bucket.
3) set source and destiny prefix for DICOM file.
4) cleanup de-id DICOM dir and evaluation DICOM dir
5) set aws session with user profile name.

In [ ]:
from med_img_de_id_class import ProcessMedImage
from common.utils import get_boto3_session, cleanup_dir, get_date_time, dump_dict_to_tsv
# setup environment
LOC_DICOM_FOLDER = '/Users/user/Documents/AI/input_data/'
LOC_DE_ID_DICOM_FOLDER = '../images/med_de_id_img/evaluation/data/'
LOC_EVAL_DICOM_FOLDER = '../images/med_eval_img/evaluation/data/'
SOURCE_BUCKET = "de-id-src"
DESTINATION_BUCKET = "de-id-dest"
SOURCE_PREFIX = "dicom-images/"
DESTINATION_PREFIX = "de-id-dicom-images/"
EVAL_BUCKET = "de-id-evl"
EVAL_PREFIX = "eval-de-id-dicom-images/"
FILE_NAME = 'file_name'
FILE_PATH = 'file_path'
FILE_PREFIX = 'prefix'

# cleanup destination dirs
cleanup_dir([LOC_DE_ID_DICOM_FOLDER, LOC_EVAL_DICOM_FOLDER ])
aws_session = None
rule_config_file_path= '../configs/de-id/de_id_rules_auto.yaml'

## De-identification On Batch DICOM files

In [ ]:
import glob, os, datetime
import warnings
# Suppress all UserWarnings
warnings.simplefilter("ignore", UserWarning)
dicom_list = []
dicom_files = glob.glob('{}/**/*.dcm'.format(LOC_DICOM_FOLDER), recursive=True)
print(f'Found {len(dicom_files)} DICOM files under {LOC_DICOM_FOLDER}')
start_date_time = datetime.datetime.now()
print(f"Start De-id on Batch of DICAM Files at {get_date_time()}")
# dicom_files = dicom_files[20000:]
process_cnt = 0
metadata_redacted_cnt = 0
pixel_redacted_cnt = 0
# create a de-id processor
aws_session = get_boto3_session("esi")
processor = ProcessMedImage(aws_session, rule_config_file_path, True)
try:
    
    last_de_if_fold = "None"
    for filepath in dicom_files:
        filename = os.path.basename(filepath)
        prefix = os.path.join(SOURCE_PREFIX, '/'.join(filepath.split('/')[-4:-2]))
        key = os.path.join(prefix, filename).replace('dcm', 'png')
        short_file_path = '/'.join(filepath.split('/')[-4:])
        text_in_image = False
        phi_in_image = False
        redacted_tags = 0
    
        result = processor.parse_dicom_file(SOURCE_BUCKET, key, filepath, True)
        dicom_dataset = processor.ds
        if not dicom_dataset:
            print(f'Error parsing dicom file, invalid metadata: {filepath}')
            continue
        if processor.image_data is None:
            print(f'Error parsing dicom file, invalid pixel data: {filepath}')
        if dicom_dataset:
            redacted_count, redacted_tags = processor.de_identify_dicom()
            metadata_redacted_cnt += 1
            id_text_detected, text_in_image = processor.detect_id_in_img(SOURCE_BUCKET, key, True)
            if text_in_image and id_text_detected and len(id_text_detected) > 0:
                phi_in_image = True
                print(f'Sensitive text detected in {filepath}')
                print (f'Found PHI in pixel: {id_text_detected} in DICOM: {short_file_path}.')
                processor.redact_id_in_image(id_text_detected)
                print('PHI in pixel have been redacted')
                pixel_redacted_cnt += 1
                print('Filename: {}, Filepath: {}, Text In Pixel: {}, PHI In Pixel: {}'.format(filename, short_file_path, text_in_image, phi_in_image))
            else:
                local_de_id_png = None
            local_de_id_dicom = f"{LOC_DE_ID_DICOM_FOLDER}{processor.patient_id}/{processor.studyInstanceUID}/{processor.seriesInstanceUID}/{filename}"
            if last_de_if_fold != os.path.dirname(local_de_id_dicom):
                last_de_if_fold = os.path.dirname(local_de_id_dicom)
                os.makedirs(os.path.dirname(local_de_id_dicom), exist_ok=True)
            processor.save_de_id_dicom(local_de_id_dicom)
            process_cnt += 1
    processor.save_mappings()
except Exception as e:
    print(f'Error processing dicom file: {e}')
    raise e
finally:
    processor.close()
    processor = None

end_date_time = datetime.datetime.now()
print(f'Completed De-in on batch of DiCAM files at {get_date_time()}')
print(f'Total {process_cnt} DICOM files are processed.')
run_time = (end_date_time - start_date_time).total_seconds()
print(f'Total run time: {run_time} seconds')

## Statistics of De-identification on Batch DICOM Dataset

In [ ]:
processed_count = process_cnt
total_dicom_files = len(dicom_files)
print(f"Number of DICOM files: {total_dicom_files}")
processed_rate = round(processed_count/len(dicom_files) * 100, 3)
print(f"Processed {processed_rate}% of DICOM files")
mean_process_time = round(run_time/total_dicom_files, 1)
print(f"Average processing time per DICOM: {mean_process_time} seconds")
redacted_count = metadata_redacted_cnt
print(f"Number of Redacted DICOM: {redacted_count}")
print(f"Redacted Ratio: {round(redacted_count/processed_count)*100}%")
redacted_metadata_count = metadata_redacted_cnt
print(f"Number of DICOM with PHI in Metadata: {redacted_metadata_count}")
print(f"Number of DICOM with PHI in Pixel: {pixel_redacted_cnt} detected by Rules")
# cleanup unused resources
dicom_list = None
dicom_files = None


In [ ]:
import sys
try:
    sys.exit(1)
except: 
    print("Completed!")